# 2. 文本向量化

- 分词后，需要把文本数值化，数值化需要把词转化为向量，把词转换为向量的方式有三种：
    - 单热编码
    - TF-IDF
    - 词袋模式

## 1.1. One-Hot编码

- 单热编码的方式比较简单：
    - 把分词的词形成词典，以词典的数量作为向量的长度，每个词对应该向量的某个下标，该向量对应下标处的值为1。词的下标位置通常由词典的中改词的词频大小决定。
    - 这种方法产生的矩阵容易是稀疏矩阵。

In [15]:
# sklearn的例子：sklearn提供了多种新式
import numpy as np
from sklearn.preprocessing import OneHotEncoder

# data = np.array(  # 支持单词
#     [['红'], ['蓝'], ['绿'], ['黄'], ['紫'], ['翠']]
# )

data = np.array( # 支持样本与标签
    [
        ['红', 'S'],
        ['蓝', 'M'],
        ['绿', 'L'],
        ['红', 'L'],
        ['蓝', 'S'],
        ['绿', 'M']
    ]
)
encoder = OneHotEncoder(
    sparse_output=False,  # 返回稠密矩阵（便于查看）
    handle_unknown='ignore'  # 忽略未知类别
)

encoded_data = encoder.fit_transform(data)
print(f"\n编码后形状: {encoded_data}")


编码后形状: [[1. 0. 0. 0. 0. 1.]
 [0. 0. 1. 0. 1. 0.]
 [0. 1. 0. 1. 0. 0.]
 [1. 0. 0. 1. 0. 0.]
 [0. 0. 1. 0. 0. 1.]
 [0. 1. 0. 0. 1. 0.]]


- 代码说明：
    - 前面三个是样本的单热编码，后面三个是标签的单热编码。
    - 单热编码自动统计词表。

In [16]:
# torch中也提供单热编码
import torch
import torch.nn.functional as F

# 假设有一组类别标签（必须是 LongTensor 类型）
labels = torch.tensor([0, 1, 2, 3, 4, 5])

# 最简单的用法：自动推断类别数（最大值+1）
one_hot = F.one_hot(labels)
print(one_hot)

tensor([[1, 0, 0, 0, 0, 0],
        [0, 1, 0, 0, 0, 0],
        [0, 0, 1, 0, 0, 0],
        [0, 0, 0, 1, 0, 0],
        [0, 0, 0, 0, 1, 0],
        [0, 0, 0, 0, 0, 1]])


- 代码说明：
    - 因为单热编码的简易性，一般只用来对标签进行向量化。

## 1.2. TF-IDF(词频-逆文档频率)

- TF-IDF(Term Frequency-Inverse Document Frequency)
    - 核心思想**一个词在一篇文档中出现的频率越高，且在整个语料库中出现的范围越广（即很多文档都有），则这个词越重要**。反之，像“的”、“了”、“是”这种几乎所有文档都有的词，会被判定为不重要。
    - 计算公式如下：
    - TF-IDF 由两部分相乘得到：$TF\text{-}IDF = TF \times IDF$
        - 词频：$TF_{i,j} = \frac{n_{i,j}}{\sum_k n_{k,j}}$
            - 衡量一个词在**当前文档**中出现的频繁程度。
        - 逆文档频率：$IDF_i = \log \frac{N}{1 + DF_i}$
            - 衡量一个词的**普遍重要性**。一个词出现在越多的文档中，说明它越普遍、越不特殊，其权重就越低。

In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

# 示例文档
documents = [
    "I like to eat apples.",
    "Apples are a type of fruit.",
    "I don't like to eat bananas.",
    "Bananas and apples are both fruits.",
]
# 创建TF-IDF向量化器
vectorizer = TfidfVectorizer()
# 拟合并转换
tfidf_matrix = vectorizer.fit_transform(documents)
# 查看结果
print(f"TF-IDF矩阵形状: {tfidf_matrix.shape}")
print(f"词汇表: {vectorizer.get_feature_names_out()}")
print(f"\nTF-IDF矩阵:\n{tfidf_matrix.toarray()}")
print("-" * 60)
vectorizer.transform(["Apples are a type of fruit."]).toarray()  # 稠密矩阵转换为稀疏矩阵

TF-IDF矩阵形状: (4, 13)
词汇表: ['and' 'apples' 'are' 'bananas' 'both' 'don' 'eat' 'fruit' 'fruits' 'like'
 'of' 'to' 'type']

TF-IDF矩阵:
[[0.         0.42344193 0.         0.         0.         0.
  0.52303503 0.         0.         0.52303503 0.         0.52303503
  0.        ]
 [0.         0.31799276 0.39278432 0.         0.         0.
  0.         0.49819711 0.         0.         0.49819711 0.
  0.49819711]
 [0.         0.         0.         0.4222466  0.         0.53556627
  0.4222466  0.         0.         0.4222466  0.         0.4222466
  0.        ]
 [0.46370919 0.29597957 0.36559366 0.36559366 0.46370919 0.
  0.         0.         0.46370919 0.         0.         0.
  0.        ]]
------------------------------------------------------------


array([[0.        , 0.31799276, 0.39278432, 0.        , 0.        ,
        0.        , 0.        , 0.49819711, 0.        , 0.        ,
        0.49819711, 0.        , 0.49819711]])

In [25]:
import jieba
from sklearn.feature_extraction.text import TfidfVectorizer

# 中文分词函数
def chinese_tokenizer(text):
    return jieba.lcut(text)

# 准备中文文档
chinese_docs = [
    "自然语言处理是人工智能的一个重要分支",
    "机器学习包括监督学习和无监督学习",
    "深度学习是机器学习的一个子领域",
    "计算机视觉让机器能够看懂图像",
    "自然语言处理和计算机视觉都是AI的重要方向"
]

# 创建中文TF-IDF向量化器
vectorizer = TfidfVectorizer(
    tokenizer=chinese_tokenizer,  # 使用jieba分词
    ngram_range=(1, 2),           # 包含unigram和bigram
    max_df=0.8,                   # 忽略高频词
    min_df=1,                     # 忽略低频词
    lowercase=False               # 中文不需要小写
)

# 转换为TF-IDF
tfidf_matrix = vectorizer.fit_transform(chinese_docs)

# 查看结果
feature_names = vectorizer.get_feature_names_out()
print(f"特征数量: {len(feature_names)}")
print(f"特征示例: {feature_names[:20]}")
print(f"\nTF-IDF矩阵形状: {tfidf_matrix.shape}")
# print(tfidf_matrix.toarray())

特征数量: 62
特征示例: ['AI' 'AI 的' '一个' '一个 子' '一个 重要' '人工智能' '人工智能 的' '分支' '包括' '包括 监督' '和'
 '和 无' '和 计算机' '图像' '处理' '处理 和' '处理 是' '子' '子 领域' '学习']

TF-IDF矩阵形状: (5, 62)


## 1.3. 词袋模型

- 词袋模型是统计形成词袋，词袋就是对词典中词编号。一般使用步骤包含：
    - 词频统计
    - 构建词袋 

In [7]:
import jieba
from collections import Counter
from torchtext.vocab import Vocab  # 构建词袋

# 准备中文文档
chinese_docs = [
    "自然语言处理是人工智能的一个重要分支",
    "机器学习包括监督学习和无监督学习",
    "深度学习是机器学习的一个子领域",
    "计算机视觉让机器能够看懂图像",
    "自然语言处理和计算机视觉都是AI的重要方向"
]

def yield_zh_tokens():
    for line in chinese_docs:
        yield jieba.lcut(line)   # 
# 构建引文的分词器
zh_toks = yield_zh_tokens()

# 使用Counter统计词频
zh_counter = Counter()
for zh_tok in zh_toks:
    zh_counter.update(zh_tok)
# 构建词袋（词袋的保存）（torchtext 0.6.0的词袋实现）
zh_vocab = Vocab(
    counter=zh_counter,
    min_freq=1,
    specials=["<s>", "</s>", '<unk>', '<pad>'],
    specials_first=True
)
line = "自然语言处理和计算机视觉都是AI的重要方向"
tokens = [zh_vocab.stoi[word]  for word in  jieba.cut(line)]
print(tokens)

[12, 10, 9, 14, 13, 29, 5, 16, 7, 15, 23]


## 1.4. 说明

- 上面技术是早期解决文本向量的技术，从统计角度出发，其实还有分布式表示，其中就包含word2vec。在Word2Vec基础上，还发展出改进算法：
    - GloVe：全局向量。结合Word2Vec的局部上下文和矩阵分解的全局统计，在词类比任务上表现优秀。
    - FastText：Word2Vec的改进版。学习的是字符级 n-gram 特征，能解决未登录词（OOV）问题（比如拼错词、新造词）。